# **LIGHT GBM OPTIMIZACIÓN BAYESIANA — AGOSTO (202108)**

## 1. SETEOS INICIALES

### 1.1  Seteo Ambiente Local

In [1]:
# Rutas locales
if (!exists("PARAM")) PARAM <- list()
PARAM$data_dir <- "C:/Users/germa/Desktop/Personal/Maestría/Data Mining/Primer Año/Data Mining en Economía y Finanzas/Clase1"

archivo_crudo    <- file.path(PARAM$data_dir, "competencia_01_crudo.csv")
archivo_ternaria <- file.path(PARAM$data_dir, "competencia_01.csv.gz")

if (!file.exists(archivo_crudo)) {
  stop("No encuentro el archivo crudo en: ", archivo_crudo)
}

### 1.2 Generacion de la clase_ternaria

In [2]:
require( "data.table" )

# leo el dataset
dataset <- fread(archivo_crudo)

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

# grabo local, en vez de /content/datasets/
fwrite( dataset,
    file = archivo_ternaria,
    sep = ","
)


Cargando paquete requerido: data.table



In [3]:
setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202103,BAJA+1,1019
202103,BAJA+2,960
202103,CONTINUA,160921
202104,BAJA+1,964
202104,BAJA+2,1139
202104,CONTINUA,161181
202105,BAJA+1,1143
202105,BAJA+2,870
202105,CONTINUA,161755


## 2 PROCESAMIENTO

### 2.1 Inicio

Limpio el ambiente de R

In [4]:
format(Sys.time(), "%a %b %d %X %Y")

# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

# vuelvo a definir la ruta local, porque el rm() de arriba borró todo
archivo_ternaria <- "C:/Users/germa/Desktop/Personal/Maestría/Data Mining/Primer Año/Data Mining en Economía y Finanzas/Clase1/competencia_01.csv.gz"

[1] "mié oct 07 01:42:05 2026"

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,724134,38.7,1446817,77.3,1446817,77.3
Vcells,1350872,10.4,170814615,1303.3,212576781,1621.9


### 2.2 Carga de Librerias

In [5]:
# Busco las librerías que necesito
library(data.table)
library(R.utils)
library(rlist)
library(bbotk)
library(mlr3)
library(mlr3mbo)
library(mlr3learners)
# library(paradox)
library(DiceKriging)
library(lightgbm)

Cargando paquete requerido: R.oo

Cargando paquete requerido: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Adjuntando el paquete: 'R.oo'


The following object is masked from 'package:R.methodsS3':

    throw


The following objects are masked from 'package:methods':

    getClasses, getMethods


The following objects are masked from 'package:base':

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Adjuntando el paquete: 'R.utils'


The following object is masked from 'package:utils':

    timestamp


The following objects are masked from 'package:base':

    cat, commandArgs, getOption, isOpen, nullfile, parse, use, warnings


Cargando paquete requerido: paradox


Adjuntando el paquete: 'mlr3'


The following object is masked from 'package:R.utils':

    resample


Cargand

### 2.3 Definición de Parametros

In [6]:
PARAM <- list()
PARAM$experimento <- "Pipeline_24"
PARAM$semilla_primigenia <- 420029

In [7]:
PARAM$training_pct <- 70
PARAM$train <- c(202103, 202104, 202105, 202106, 202107, 202108)  # usado solo para filtrar el panel completo
PARAM$train_modelo_A <- c(202103, 202104, 202105, 202106)         # meses con clase 100% real
PARAM$train_final <- c(202103, 202104, 202105, 202106, 202107, 202108)
PARAM$future <- c(202108)
PARAM$cortes <- seq(9000, 15000, by = 500)

In [8]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.1

In [9]:
# Parametros LightGBM

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "average_precision",
  feature_pre_filter= FALSE,
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  force_row_wise= TRUE,
  deterministic= TRUE,
  verbosity= -100,
  num_threads= 2,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L,
  min_gain_to_split= 0,
  lambda_l1= 0.0,
  lambda_l2= 0.0,
  max_bin= 31L,

  bagging_fraction= 1.0,
  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  drop_rate= 0.1,
  max_drop= 50,
  skip_drop= 0.5,

  extra_trees= FALSE,

  early_stopping= 0,
  min_data_in_leaf= 0,     # fijo, esto es FUNDAMENTAL
  feature_fraction= 0.50,  # fijo
  learning_rate= 0.005,    # fijo

  num_iterations= 20000,     # se modificara
  num_leaves= 1024,          # se modificara
  min_sum_hessian_in_leaf= 1e-06  # se modificara
)


In [10]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}

### 2.4  Preprocesamiento

In [11]:
PARAM$exp_dir <- "C:/Users/germa/Desktop/Personal/Maestría/Data Mining/Primer Año/Data Mining en Economía y Finanzas/Competencia_1_Final"
PARAM$bo_dir  <- "C:/Users/germa/Desktop/Personal/Maestría/Data Mining/Primer Año/Data Mining en Economía y Finanzas/Competencia_1_Final/BO/BO_Exp24_agosto"

setwd(PARAM$exp_dir)
dir.create(PARAM$experimento, showWarnings = FALSE)
setwd(file.path(PARAM$exp_dir, PARAM$experimento))

In [12]:
# lectura del dataset
dataset <- fread(archivo_ternaria, stringsAsFactors = TRUE)

In [13]:
# LAG 1, LAG 2 y sus DELTA -- sobre TODO el dataset, antes de filtrar por PARAM$train/future
# delta_k = valor actual - lag_k. lag2 NA para 202103/202104 (sin historia 2 meses
# atras en el panel); real desde 202105 en adelante.
setorder(dataset, numero_de_cliente, foto_mes)

campos_lagueables <- setdiff(colnames(dataset),
                              c("numero_de_cliente", "foto_mes", "clase_ternaria",
                                "cprestamos_personales", "mprestamos_personales"))

dataset[, paste0(campos_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
        by = numero_de_cliente, .SDcols = campos_lagueables]
dataset[, paste0(campos_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
        by = numero_de_cliente, .SDcols = campos_lagueables]

for (campo in campos_lagueables) {
  dataset[, paste0(campo, "_delta1") := get(campo) - get(paste0(campo, "_lag1"))]
  dataset[, paste0(campo, "_delta2") := get(campo) - get(paste0(campo, "_lag2"))]
}

In [14]:
# ===== campos_buenos: solo depende de nombres de columna =====
campos_buenos <- setdiff(
  colnames(dataset),
  c("clase_ternaria", "clase01", "clase01_conocida", "prob_modelo_A", "es_sintetico",
    "fold", "azar", "training")
)
campos_buenos <- setdiff(campos_buenos, c("cprestamos_personales", "mprestamos_personales"))

# ===== universo CONOCIDO: 203-206 completos + de 207 solo los BAJA+1 =====
dataset_conocido <- dataset[
  foto_mes %in% PARAM$train_modelo_A |
  (foto_mes == 202107 & clase_ternaria == "BAJA+1")
]
dataset_conocido[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1","BAJA+2"), 1L, 0L)]

particionar(dataset_conocido,
  division = c(PARAM$training_pct, 100L - PARAM$training_pct),
  agrupa = "clase_ternaria",
  seed = PARAM$semilla_primigenia
)

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_conocido[, azar := runif(.N)]
dataset_conocido[, training := 0L]
dataset_conocido[
  fold == 1 & (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1","BAJA+2")),
  training := 1L
]

# ===== Modelo A: SOLO los meses 100% reales, con los hiperparametros razonables de Exp_01 =====
param_razonable <- modifyList(PARAM$lgbm$param_fijos, list(
  feature_fraction        = 0.81828,
  learning_rate           = 0.01222983,
  num_iterations          = 2530,
  num_leaves              = 63,
  min_sum_hessian_in_leaf = 0.06232269
))

dtrain_modelo_A <- lgb.Dataset(
  data  = data.matrix(dataset_conocido[training == 1L & foto_mes %in% PARAM$train_modelo_A, campos_buenos, with = FALSE]),
  label = dataset_conocido[training == 1L & foto_mes %in% PARAM$train_modelo_A, clase01],
  free_raw_data = FALSE
)
modelo_A <- lgb.train(data = dtrain_modelo_A, param = param_razonable)

# ===== universo DESCONOCIDO: resto de 207 + todo 208 (no tiene clase calculable) =====
dataset_desconocido <- dataset[
  (foto_mes == 202107 & clase_ternaria != "BAJA+1") | foto_mes == 202108
]
dataset_desconocido[, prob_modelo_A := predict(modelo_A, data.matrix(dataset_desconocido[, campos_buenos, with = FALSE]))]

# ===== prob_corte calibrado contra la proporcion historica real de los meses conocidos =====
proporcion_historica <- dataset[foto_mes %in% PARAM$train_modelo_A, mean(clase_ternaria %in% c("BAJA+1","BAJA+2"))]
prob_corte <- quantile(dataset_desconocido$prob_modelo_A, probs = 1 - proporcion_historica)
cat("Proporcion historica BAJA:", proporcion_historica, "\n")
cat("prob_corte:", prob_corte, "\n")

dataset_desconocido[, clase01_conocida := ifelse(prob_modelo_A > prob_corte, 1L, 0L)]
dataset_desconocido[, es_sintetico := TRUE]

dataset_conocido[, clase01_conocida := clase01]
dataset_conocido[, es_sintetico := FALSE]

Proporcion historica BAJA: 0.01233362 
prob_corte: 0.735214 


### 3. PRODUCCIÓN

In [15]:
# cargo los mejores hiperparametros de la BO correspondiente (BO_log.txt compartido
# cuando dos pipelines usan la misma estructura de dataset)
tb_bayesiana <- fread(file.path(PARAM$bo_dir, "BO_log.txt"))
setorder( tb_bayesiana, -metrica )
PARAM$out$lgbm$mejores_hiperparametros <- as.list(tb_bayesiana[1])

cat("Mejor hiperparametro usado (de", nrow(tb_bayesiana), "filas en BO_log.txt):\n")
print(tb_bayesiana[1, .(num_iterations, num_leaves, min_sum_hessian_in_leaf, feature_fraction, learning_rate, metrica)])

Mejor hiperparametro usado (de 25 filas en BO_log.txt):
   num_iterations num_leaves min_sum_hessian_in_leaf feature_fraction
            <int>      <int>                   <num>            <num>
1:           2671         92              0.00154167        0.9303991
   learning_rate   metrica
           <num>     <num>
1:    0.04513209 0.2775877


In [16]:
dataset_train <- rbindlist(list(dataset_conocido, dataset_desconocido), use.names = TRUE, fill = TRUE)
dataset_train <- dataset_train[foto_mes %in% PARAM$train_final]
dataset_train[, .N, list(foto_mes, es_sintetico)]

foto_mes,es_sintetico,N
<int>,<lgl>,<int>
202103,FALSE,162900
202104,FALSE,163284
202105,FALSE,163768
202106,FALSE,164114
202107,FALSE,1103
202107,TRUE,163245
202108,TRUE,164647


In [17]:
# ===== SEMILLERIO: 10 semillas para el submit (el bot promedia la ganancia entre ellas) =====
param_final <- modifyList(PARAM$lgbm$param_fijos, PARAM$out$lgbm$mejores_hiperparametros)

dfuture <- dataset[foto_mes %in% PARAM$future]

PARAM$semillerio <- c(420029, 425207, 420521, 420307, 420929,
                       410029, 430029, 440029, 450029, 460029)
PARAM$cortes_submit <- c(10000, 10500, 11000, 11500, 12000, 12500, 13000, 13500, 14000)

predicciones <- matrix(nrow = nrow(dfuture), ncol = length(PARAM$semillerio))

for (i in seq_along(PARAM$semillerio)) {

  param_final$seed <- PARAM$semillerio[i]

  dtrain_final <- lgb.Dataset(
    data  = data.matrix(dataset_train[, campos_buenos, with = FALSE]),
    label = dataset_train[, clase01_conocida]
  )

  modelo_final <- lgb.train(
    data  = dtrain_final,
    param = param_final
  )

  predicciones[, i] <- predict(modelo_final, data.matrix(dfuture[, campos_buenos, with = FALSE]))

  cat("semilla", PARAM$semillerio[i], "lista\n")
}

# ===== Exportacion de CSVs para el submit (sin header, sin notacion cientifica) =====
# por cada corte, un submit con las 10 semillas (1 archivo por semilla)
dir.create("CSV", showWarnings = FALSE)
options(scipen = 999)
clientes <- as.integer(dfuture$numero_de_cliente)

for (i in seq_along(PARAM$semillerio)) {
  orden <- order(-predicciones[, i])
  clientes_ordenados <- clientes[orden]
  for (corte in PARAM$cortes_submit) {
    archivo <- file.path("CSV", sprintf("corte_%d_semilla_%d.csv", corte, PARAM$semillerio[i]))
    fwrite(
      data.table(numero_de_cliente = clientes_ordenados[1:corte]),
      file = archivo, col.names = FALSE
    )
  }
}
cat("CSVs generados en:", file.path(getwd(), "CSV"), "\n")
cat("Total archivos:", length(PARAM$semillerio) * length(PARAM$cortes_submit), "\n")

semilla 420029 lista
semilla 425207 lista
semilla 420521 lista
semilla 420307 lista
semilla 420929 lista
semilla 410029 lista
semilla 430029 lista
semilla 440029 lista
semilla 450029 lista
semilla 460029 lista
CSVs generados en: C:/Users/germa/Desktop/Personal/Maestría/Data Mining/Primer Año/Data Mining en Economía y Finanzas/Competencia_1_Final/Pipeline_24/CSV 
Total archivos: 90 


In [18]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "mié oct 07 05:51:52 2026"